# Un agente ReAct mínimo, paso a paso

**Facsímil 5 · Agentes y orquestación** — capítulos 2, 3 y 5
(qué es un agente: estado, acción y observación; *tools*; de ReAct a sistemas multiagente).

Un agente no es «un prompt más listo». Es un **bucle** alrededor del modelo: decidir una acción,
ejecutarla, mirar el resultado y volver a decidir, hasta resolver la tarea. En este cuaderno construyes
el esqueleto de ese bucle —el patrón **ReAct** (*razonar y actuar*)— y lo ves resolver tareas que no se
pueden contestar de un tirón, usando herramientas de verdad y dejando una **traza** auditable de su
razonamiento. El «modelo» que decide la siguiente acción lo simulamos con reglas, para que el bucle
—que es lo importante— se vea sin caja negra. Entender ese ciclo es entender por qué un agente puede
encadenar pasos... y por qué a veces se va por las ramas.

### Qué vas a aprender
- Qué distingue a un **agente** de una sola llamada a un modelo, y cuándo merece la pena.
- El **formato de texto** de ReAct (pensamiento, acción, observación) y cómo se **parsea** sin peligro.
- El bucle **ReAct** completo: pensar → actuar → observar, repetido hasta tener la respuesta.
- Qué son las **herramientas** (*tools*) y cómo añadir más sin tocar el bucle.
- Por qué los **límites** (tope de pasos, detección de bucles, reacción a errores) separan un agente útil
  de uno que se descontrola.
- Cómo la **traza** se convierte en observabilidad, y cómo se pasa de **un** agente a **varios** coordinados.

### Cuánto cuesta
Unos 16 minutos. CPU, sin claves (el «modelo» se simula con reglas).


> **Inteligencia artificial para gente curiosa** · facsímil interactivo
> 
> Web del facsímil: https://www.iaparagentecuriosa.686f6c61.dev/ · Autor: @686f6c61 · Fecha: 2026-06-26 · Versión 1.0
> 
> Este cuaderno acompaña al facsímil: ejecútalo de arriba abajo, lee cada celda de texto
> antes de correr la de código y detente en las salidas. La gracia no es que «salga», sino
> entender *por qué* sale.

In [ ]:
import ast, operator
# Las herramientas: funciones normales que el agente puede invocar.
CATALOGO = {"teclado": 29.90, "raton": 15.50, "monitor": 149.00, "webcam": 39.00}
STOCK    = {"teclado": 12, "raton": 0, "monitor": 5, "webcam": 8}   # el raton esta agotado
PEDIDOS  = {"1002": "en reparto, llega mañana", "1003": "entregado el 24/05"}
CLIMA    = {"madrid": "18 grados y soleado", "sevilla": "26 grados y despejado", "bilbao": "15 grados y lluvia"}

# Calculadora SEGURA con ast (nunca eval, que ejecutaria codigo arbitrario).
_OPS = {ast.Add: operator.add, ast.Sub: operator.sub,
        ast.Mult: operator.mul, ast.Div: operator.truediv, ast.USub: operator.neg}
def _ev(n):
    if isinstance(n, ast.Constant) and isinstance(n.value, (int, float)): return n.value
    if isinstance(n, ast.BinOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.left), _ev(n.right))
    if isinstance(n, ast.UnaryOp) and type(n.op) in _OPS: return _OPS[type(n.op)](_ev(n.operand))
    raise ValueError("expresion no permitida")

def calculadora(expr):     return _ev(ast.parse(expr, mode="eval").body)
def buscar_precio(p):      return CATALOGO.get(p.lower(), "no encontrado")
def stock(p):              return STOCK.get(p.lower(), "no en catalogo")
def estado_pedido(n):      return PEDIDOS.get(str(n), "pedido desconocido")
def tiempo(ciudad):        return CLIMA.get(ciudad.lower(), "ciudad desconocida")

TOOLS = {"calculadora": calculadora, "buscar_precio": buscar_precio,
         "stock": stock, "estado_pedido": estado_pedido, "tiempo": tiempo}
print("Herramientas disponibles:", ", ".join(TOOLS))


## 1. Agente o prompt: cuándo merece la pena

Para muchas tareas, una sola llamada al modelo basta: «resume este texto», «traduce esta frase». Pero
otras requieren **varios pasos que dependen unos de otros** y **tocar el mundo** (buscar un dato,
calcular, consultar un sistema). Ahí entra un agente: un bucle que decide qué hacer, lo hace, mira el
resultado y sigue. La regla práctica: usa un agente solo cuando la tarea sea multi-paso, no se pueda
resolver de un tirón, y el coste de la complejidad compense. Nuestra tarea de ejemplo es justo así:
«quiero 3 teclados, cuánto es, hay stock y cómo va mi pedido 1002» mezcla un cálculo, dos búsquedas y
una consulta, y cada paso depende del anterior.


## 2. El formato ReAct, y cómo parsearlo sin peligro

En un agente real, el modelo no «llama» a la herramienta: **escribe texto** con un formato pactado, algo
como `Pensamiento: ...` seguido de `Acción: buscar_precio[teclado]`. Tu código lee ese texto, **extrae**
el nombre de la herramienta y su argumento, y lo ejecuta. Ese paso de extracción es delicado: hay que
hacerlo con un parseo **acotado** (una expresión regular que solo acepta `nombre[argumento]`), nunca con
`eval`, que ejecutaría cualquier cosa que el modelo escribiera. Empezamos por ahí: la pieza que traduce
texto en una acción concreta.


In [ ]:
import re
# Solo aceptamos el formato exacto nombre[argumento]. Nada de eval: parseo acotado.
PATRON_ACCION = re.compile(r"^Acción:\s*(\w+)\[(.*)\]\s*$")

def parsear_accion(texto):
    # Devuelve (herramienta, argumento) si hay una linea 'Acción: ...', o None si no la hay.
    for linea in texto.splitlines():
        m = PATRON_ACCION.match(linea.strip())
        if m:
            return m.group(1), m.group(2)
    return None

ejemplos = [
    "Pensamiento: necesito el precio.\nAcción: buscar_precio[teclado]",
    "Pensamiento: ya tengo todo.\nRespuesta final: tres teclados cuestan 89.70 EUR.",
    "Acción: borrar_disco[/]; rm -rf /",   # texto hostil: el patron NO lo acepta como accion valida
]
for e in ejemplos:
    print(repr(parsear_accion(e)))


La primera línea da `('buscar_precio', 'teclado')`. La segunda, que solo tiene una respuesta final,
da `None`: es la señal de que el agente ha terminado. La tercera, un texto que intenta colar una orden
peligrosa, **no** encaja con `nombre[argumento]`, así que también da `None`: la herramienta
`borrar_disco` ni siquiera existe en `TOOLS` y nunca se ejecutaría. El parseo acotado es la primera línea
de defensa entre lo que el modelo *escribe* y lo que tu sistema *hace*.


## 3. El bucle ReAct completo

ReAct alterna **razonar** y **actuar**. En cada vuelta, el agente escribe un **pensamiento** (qué le
falta), elige una **acción** (qué herramienta y con qué argumento), y recibe una **observación** (el
resultado), que guarda en su **memoria**. Repite hasta tener todo para responder. Aquí el «cerebro» que
decide la siguiente acción es una función con reglas; en un agente real, esa decisión la toma el modelo,
pero **el bucle es idéntico**. Fíjate en que el cerebro mira la memoria para no repetir trabajo.


In [ ]:
def cerebro_compra(tarea, memoria):
    h = dict(memoria)
    if "precio_teclado" not in h:
        return "Pensamiento: necesito el precio unitario del teclado.\nAcción: buscar_precio[teclado]"
    if "total" not in h:
        return f"Pensamiento: con el precio ({h['precio_teclado']}) calculo 3 unidades.\nAcción: calculadora[3 * {h['precio_teclado']}]"
    if "stock_teclado" not in h:
        return "Pensamiento: antes de prometer nada, confirmo que hay existencias.\nAcción: stock[teclado]"
    if "pedido" not in h:
        return "Pensamiento: ya solo me queda el estado del pedido 1002.\nAcción: estado_pedido[1002]"
    return (f"Pensamiento: tengo precio, total, stock y estado; puedo responder.\n"
            f"Respuesta final: tres teclados cuestan {h['total']:.2f} EUR (hay {h['stock_teclado']} en stock); "
            f"tu pedido 1002 está {h['pedido']}.")

def clave_memoria(nombre, arg):
    if nombre == "buscar_precio": return f"precio_{arg.lower()}"
    if nombre == "stock":         return f"stock_{arg.lower()}"
    if nombre == "tiempo":        return f"tiempo_{arg.lower()}"
    if nombre == "calculadora":   return "total"
    if nombre == "estado_pedido": return "pedido"
    return nombre

def agente(tarea, cerebro, max_pasos=8, verbose=True):
    memoria, traza = [], []
    if verbose: print(f"TAREA: {tarea}\n")
    for paso in range(1, max_pasos + 1):
        texto = cerebro(tarea, memoria)
        pensamiento = next((l.split(":", 1)[1].strip() for l in texto.splitlines()
                            if l.strip().startswith("Pensamiento:")), "")
        accion = parsear_accion(texto)
        if accion is None:                       # no hay 'Acción:' -> es la respuesta final
            final = next((l.split(":", 1)[1].strip() for l in texto.splitlines()
                          if l.strip().startswith("Respuesta final:")), "")
            if verbose: print(f"Paso {paso}\n  Pensamiento: {pensamiento}\n  RESPUESTA: {final}")
            traza.append({"paso": paso, "pensamiento": pensamiento, "accion": None, "observacion": final})
            return traza
        nombre, arg = accion
        obs = TOOLS[nombre](arg)
        memoria.append((clave_memoria(nombre, arg), obs))
        traza.append({"paso": paso, "pensamiento": pensamiento, "accion": f"{nombre}[{arg}]", "observacion": obs})
        if verbose:
            print(f"Paso {paso}\n  Pensamiento: {pensamiento}\n  Acción: {nombre}[{arg}]\n  Observación: {obs}\n")
    if verbose: print("(se acabaron los pasos sin respuesta final)")
    return traza

_ = agente("Quiero 3 teclados, cuánto es, hay stock y cómo va mi pedido 1002?", cerebro_compra)


**Mira la traza.** El agente no contestó de golpe: descompuso la tarea en pasos, usó una herramienta
distinta en cada uno y fue **acumulando observaciones** en su memoria hasta poder responder. Ese ir y
venir —decidir, ejecutar, mirar— es exactamente lo que hace un agente real; lo único que cambia es que
ahí el modelo elige la acción en vez de nuestras reglas. Y la traza (pensamiento / acción / observación)
hace cada paso **auditable**: puedes ver *por qué* hizo lo que hizo.


## 4. Más herramientas, el mismo bucle

Aquí está la gracia del patrón: para que el agente sepa hacer cosas nuevas, **no se toca el bucle**, solo
se añaden herramientas y se enseña al cerebro a usarlas. Cambiamos de tarea por completo —«¿qué tiempo
hace en Bilbao y cuántos monitores quedan?»— con un cerebro distinto, pero el mismo `agente()` de antes.


In [ ]:
def cerebro_consulta(tarea, memoria):
    h = dict(memoria)
    if "tiempo_bilbao" not in h:
        return "Pensamiento: primero compruebo el tiempo en Bilbao.\nAcción: tiempo[bilbao]"
    if "stock_monitor" not in h:
        return "Pensamiento: ahora miro cuántos monitores quedan.\nAcción: stock[monitor]"
    return (f"Pensamiento: ya tengo las dos cosas.\n"
            f"Respuesta final: en Bilbao hace {h['tiempo_bilbao']}; quedan {h['stock_monitor']} monitores.")

_ = agente("¿Qué tiempo hace en Bilbao y cuántos monitores quedan?", cerebro_consulta)


**El bucle no se ha enterado.** `agente()` es el mismo; lo único nuevo son las herramientas `tiempo` y
`stock` y un cerebro que sabe cuándo usarlas. Esa separación —un motor genérico, un conjunto de
herramientas, una política de decisión— es lo que hace a los agentes extensibles: añadir una capacidad es
añadir una función, no reescribir el sistema.


## 5. Por qué esto no es un juguete: el tope de pasos

El bucle tiene un riesgo real. Si el «cerebro» se equivoca al elegir la acción, el agente puede quedarse
**dando vueltas** sin avanzar, gastando tiempo y dinero. Por eso los agentes serios ponen barreras. La
más básica: un **tope de pasos**. Simulamos un cerebro roto que nunca termina y vemos cómo el tope lo
corta.


In [ ]:
def cerebro_roto(tarea, memoria):
    return "Pensamiento: ...otra vez lo mismo, no aprendo.\nAcción: buscar_precio[teclado]"

pasos, TOPE, mem = 0, 5, []
while pasos < TOPE:                      # SIN este tope, seria un bucle infinito
    texto = cerebro_roto("", mem); pasos += 1
    if parsear_accion(texto) is None: break
print(f"El agente con cerebro roto se cortó a los {pasos} pasos gracias al tope.")
print("Sin ese límite, habría llamado a la herramienta para siempre, gastando dinero.")
print("El tope de pasos es la red de seguridad más básica (y más importante) de todo agente.")


## 6. Detectar bucles: no repetir la misma acción

El tope de pasos corta, pero tarde: gasta cinco llamadas antes de rendirse. Una barrera más fina es
**detectar la repetición**: si el agente propone una acción que ya hizo, con el mismo argumento, casi
seguro está atascado y conviene pararlo ya. Lo añadimos como una pequeña guarda alrededor del bucle.


In [ ]:
def agente_con_guarda(tarea, cerebro, max_pasos=6):
    memoria, vistas = [], set()
    for paso in range(1, max_pasos + 1):
        texto = cerebro(tarea, memoria)
        accion = parsear_accion(texto)
        if accion is None:
            print(f"Paso {paso}: respuesta final, todo en orden."); return
        if accion in vistas:
            print(f"Paso {paso}: la acción {accion[0]}[{accion[1]}] ya se intentó; corto para no entrar en bucle.")
            return
        vistas.add(accion)
        obs = TOOLS[accion[0]](accion[1])
        memoria.append((clave_memoria(*accion), obs))
        print(f"Paso {paso}: {accion[0]}[{accion[1]}] -> {obs}")

agente_con_guarda("da igual la tarea", cerebro_roto)


**Se cortó en el segundo paso**, no en el quinto: en cuanto el cerebro repitió `buscar_precio[teclado]`,
la guarda lo detectó. Combinar un tope duro (nunca más de N pasos) con detección de repetición (no
insistas en lo que ya falló) es lo que evita que un agente se quede en un bucle caro. Son baratas de
poner y se agradecen el día que el cerebro se equivoca.


## 7. Reaccionar a las malas observaciones

Un agente bueno no solo encadena pasos: **reacciona** cuando una herramienta devuelve algo inesperado.
Si el catálogo no tiene el producto, la observación es «no encontrado», y el agente debería cambiar de
plan en vez de seguir como si nada. Lo vemos con un cerebro que detecta el problema y responde con
honestidad en vez de inventarse un precio.


In [ ]:
def cerebro_robusto(tarea, memoria):
    h = dict(memoria)
    if "precio_auriculares" not in h:
        return "Pensamiento: busco el precio de los auriculares.\nAcción: buscar_precio[auriculares]"
    if h["precio_auriculares"] == "no encontrado":
        return ("Pensamiento: el catálogo devolvió 'no encontrado'; no me invento un precio.\n"
                "Respuesta final: lo siento, los auriculares no están en el catálogo.")
    return f"Pensamiento: tengo el precio.\nRespuesta final: los auriculares cuestan {h['precio_auriculares']} EUR."

_ = agente("¿Cuánto cuestan unos auriculares?", cerebro_robusto)


**Eso es robustez.** El agente preguntó por un producto que no existe, recibió «no encontrado» y, en
vez de alucinar un precio, lo admitió. Un agente que ignora las malas observaciones es peligroso: actúa
sobre datos que no tiene. Reaccionar a lo que el mundo te devuelve es parte del oficio, y por eso la
observación viaja siempre de vuelta al cerebro antes de la siguiente decisión.


## 8. La traza como objeto: el primer paso de la observabilidad

Hasta ahora hemos *impreso* la traza. Pero `agente()` también la **devuelve** como una lista de
diccionarios. Eso es oro: con la traza como dato puedes contar pasos, ver qué herramientas se usaron,
medir y depurar sin leer un muro de texto. Es, en pequeño, la observabilidad de un agente real (facsímil
6). Ejecutamos en silencio y resumimos la trayectoria.


In [ ]:
traza = agente("Quiero 3 teclados, cuánto es, hay stock y cómo va mi pedido 1002?",
                cerebro_compra, verbose=False)
print(f"La trayectoria tuvo {len(traza)} pasos.")
herramientas = [p["accion"].split("[")[0] for p in traza if p["accion"]]
print(f"Acciones ejecutadas ({len(herramientas)}):", ", ".join(herramientas))
print("Respuesta final:", traza[-1]["observacion"])
print("\nCon la traza como objeto, medir un agente es contar y filtrar, no leer logs a ojo.")


## 9. De un agente a varios: orquestación

Cuando una tarea mezcla mundos muy distintos, en vez de un cerebro que lo sepa todo se usan **varios
agentes especialistas** y un **coordinador** que reparte el trabajo (capítulo 5). Cada especialista hace
una cosa bien y el coordinador compone la respuesta. Es la misma idea de descomponer, llevada un nivel
más arriba: ya no descompones en pasos, sino en *roles*.


In [ ]:
def agente_compras(producto, unidades):
    precio = buscar_precio(producto)
    total = calculadora(f"{unidades} * {precio}")
    return f"{unidades} x {producto} = {total:.2f} EUR (stock: {stock(producto)})"

def agente_logistica(numero):
    return f"pedido {numero}: {estado_pedido(numero)}"

def coordinador(subtareas):
    print("Coordinador: reparto el trabajo entre especialistas.\n")
    for tipo, args in subtareas:
        if tipo == "compras":
            print("  [compras]   ", agente_compras(*args))
        elif tipo == "logistica":
            print("  [logística] ", agente_logistica(*args))

coordinador([("compras", ("teclado", 3)),
             ("compras", ("monitor", 2)),
             ("logistica", ("1002",))])


**Cada especialista resolvió su parte** y el coordinador las juntó. En sistemas reales, el coordinador
suele ser otro agente que decide a quién delegar según la tarea, y cada especialista tiene su propio
bucle ReAct y sus propias herramientas. Pero la lógica es la que ya conoces: descomponer, delegar,
componer. Cuidado: más agentes también es más coste, más latencia y más sitios donde algo puede fallar;
no siempre compensa, y saber cuándo basta con uno es parte del criterio.


## 10. Pruébalo tú

1. **Una herramienta nueva sin tocar el bucle.** Añade `horario(tienda)` con un diccionario, mete su
   entrada en `TOOLS` y escribe un cerebro que la use. Comprueba que `agente()` no necesita ni un cambio.
2. **Baja el tope** a `max_pasos=2` y lanza `cerebro_compra`: se queda sin pasos antes de responder. Así
   se ve que el límite es un equilibrio entre seguridad y capacidad.
3. **Un cerebro que pide algo agotado.** Haz que consulte `stock[raton]` (devuelve 0) y que reaccione
   ofreciendo una alternativa en vez de prometer lo imposible.
4. **Rompe el formato.** Haz que un cerebro devuelva `Accion: buscar_precio(teclado)` (paréntesis, sin
   tilde). El parseo da `None` y el agente lo trata como respuesta vacía: por eso el formato pactado
   importa tanto.
5. **Amplía el coordinador** con un tercer especialista (`agente_tiempo`) y una subtarea de clima.


## 11. Errores comunes

- **Pensar que un agente es magia.** Es un bucle con herramientas. La inteligencia está tanto en el
  modelo como en el **andamiaje** (las tools, el parseo, los límites, la traza).
- **Ejecutar lo que el modelo escribe sin parsear con cuidado.** Un parseo laxo (o, peor, `eval`) deja
  que el texto del modelo se convierta en cualquier orden. Acota el formato y valida.
- **No poner tope de pasos ni detectar repeticiones.** Un agente sin freno puede entrar en bucle y gastar
  tiempo y dinero sin avanzar.
- **Ignorar las malas observaciones.** Si una herramienta falla y el agente sigue como si nada, actúa
  sobre datos inventados.
- **No dejar traza.** Sin registro de pensamiento / acción / observación, es imposible depurar por qué el
  agente hizo lo que hizo, ni medir si lo hace bien.
- **Meter varios agentes porque suena moderno.** Más roles es más coste y más fallos posibles; usa
  orquestación solo cuando un único bucle se queda corto.


## 12. Qué te llevas

- Un agente es un **bucle** (pensar, actuar, observar), no un prompt mágico. La inteligencia está tanto
  en el modelo como en el andamiaje que lo rodea.
- El modelo se comunica con **texto** en un formato pactado; tu código lo **parsea de forma acotada** y
  decide qué se ejecuta. Esa frontera es donde mandas tú.
- Las **herramientas** son funciones normales; añadir capacidades es añadir funciones, no reescribir el
  bucle. La **traza** hace auditable y medible cada paso.
- Los **límites** (tope, detección de bucles, reacción a errores) son lo que separa un agente útil de uno
  que se descontrola. Y cuando un bucle se queda corto, se **orquestan varios**.

**Para seguir:** el siguiente cuaderno formaliza cómo el modelo «pide» una herramienta (*function
calling*) y cómo validas esa petición antes de ejecutar; el capítulo 10 mide la trayectoria completa de
un agente; y el facsímil 6 lleva la traza a la observabilidad en producción.


---

### Ficha del cuaderno

- **Obra:** *Inteligencia artificial para gente curiosa* (facsímil interactivo).
- **Web:** https://www.iaparagentecuriosa.686f6c61.dev/
- **Autor:** @686f6c61
- **Fecha:** 2026-06-26
- **Versión:** 1.0

*Material pedagógico. Las salidas que ves son reales: se generan al ejecutar el código, no están escritas a mano. Si cambias algo, cambiarán: esa es la idea.*